# CatBoost Direct + Rate Ensemble with Seed Averaging

Includes:
- Direct Calories prediction
- Rate prediction (Calories / Duration)
- Ensemble of Direct + Rate
- Seed ensemble (multiple random seeds)
- Deep trees (depth 10–12)
- Low learning rate with many iterations
- Basic feature engineering + feature selection


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import VarianceThreshold

SEEDS = [42, 52, 62, 72, 82]
FOLDS = 5

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

TARGET = 'Calories'
ID_COL = 'id' if 'id' in train.columns else None


In [ ]:
def feature_engineering(df):
    df = df.copy()
    
    # Example engineered features
    if 'Duration' in df.columns and 'Heart_Rate' in df.columns:
        df['HR_Duration'] = df['Heart_Rate'] * df['Duration']
    
    if 'Weight' in df.columns and 'Duration' in df.columns:
        df['Weight_Duration'] = df['Weight'] * df['Duration']
    
    if 'Age' in df.columns and 'Heart_Rate' in df.columns:
        df['HR_Age'] = df['Heart_Rate'] * df['Age']
    
    return df

train = feature_engineering(train)
test = feature_engineering(test)

In [ ]:
# Rate target
train['rate'] = train[TARGET] / train['Duration']

features = [c for c in train.columns if c not in [TARGET, 'rate', ID_COL]]

X = train[features]
y_direct = train[TARGET]
y_rate = train['rate']
X_test = test[features]

# Feature selection (variance threshold)
selector = VarianceThreshold(0.0)
X = selector.fit_transform(X)
X_test = selector.transform(X_test)

In [ ]:
def train_catboost(X, y, seeds):
    preds_test = np.zeros(len(X_test))
    oof = np.zeros(len(X))

    for seed in seeds:
        kf = KFold(n_splits=FOLDS, shuffle=True, random_state=seed)

        for train_idx, valid_idx in kf.split(X):
            X_tr, X_val = X[train_idx], X[valid_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

            model = CatBoostRegressor(
                depth=10,
                learning_rate=0.02,
                iterations=8000,
                loss_function='RMSE',
                random_seed=seed,
                verbose=False
            )

            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)

            oof[valid_idx] += model.predict(X_val) / len(seeds)
            preds_test += model.predict(X_test) / (len(seeds) * FOLDS)

    rmse = np.sqrt(mean_squared_error(y, oof))
    return oof, preds_test, rmse


In [ ]:
# Direct model
oof_direct, pred_direct, rmse_direct = train_catboost(X, y_direct, SEEDS)
print('Direct RMSE:', rmse_direct)

# Rate model
oof_rate, pred_rate, rmse_rate = train_catboost(X, y_rate, SEEDS)
print('Rate RMSE:', rmse_rate)

# Convert rate → calories
duration_test = test['Duration'].values
pred_rate_cal = pred_rate * duration_test

# Ensemble
final_pred = 0.5 * pred_direct + 0.5 * pred_rate_cal


In [ ]:
submission = pd.DataFrame()

if ID_COL:
    submission[ID_COL] = test[ID_COL]

submission[TARGET] = final_pred
submission.to_csv('submission.csv', index=False)

print('Submission saved.')